# PDF question-answering with local Ollama models

Uses Ollama locally for embeddings and answers. Start Ollama and run `ollama pull embeddinggemma` and `ollama pull gemma3:1b` in a terminal first. The old OpenAI call failed because the API account had no credits. This version removes OpenAI and Chroma and searches vectors in memory. Run cells from top to bottom.


In [1]:
# Install PDF parsing support
%pip install -q pypdf


Note: you may need to restart the kernel to use updated packages.


Ollama setup


Ollama is local; no OpenAI API key is needed.


3.Upload the pdf

In [2]:
from pathlib import Path
pdf_path = Path(input("Enter the full path to the PDF file: ").strip().strip('"').strip("'")).expanduser()
if not pdf_path.is_file(): raise FileNotFoundError(f"PDF not found: {pdf_path}")
if pdf_path.suffix.lower() != ".pdf": raise ValueError("Please select a PDF file.")
print("Selected PDF:", pdf_path)


Selected PDF: C:\Users\arockia ludson\Downloads\USA_Employee_Handbook-Freely_Available.pdf


4.pdf reader

The `pypdf` package was installed in the setup cell above.


In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

pages = []
for page_number,page in enumerate(reader.pages, start=1):
  page_text = page.extract_text() or ""
  pages.append({
      "page": page_number,
      "text": page_text
  })

text = "\n".join(page["text"] for page in pages)
print("pages:", len(pages))
print("characters extracted:", len(text))
print("\nPreview:\n")
print(text[:2000])

pages: 34
characters extracted: 59563

Preview:

 
 
Employee Handbook 
Welcome 4 
Getting to know our company 4 
Employment basics 5 
Employment contract types 5 
Equal opportunity employment 5 
Recruitment and selection process 6 
Background checks 6 
Referrals 7 
Attendance 8 
Workplace policies 8 
Confidentiality and data protection 8 
Harassment and violence 9 
Workplace harassment 10 
Workplace violence 10 
Workplace safety and health 11 
Preventative action 12 
Emergency management 12 
Smoking 12 
Drug-free workplace 13 
Employee Code of Conduct 14 
Dress code 14 
Cyber security and digital devices 14 
Internet usage 15 
Cell phone 15 
Corporate email 16 
Social media 16 
Conflict of interest 17 
Employee relationships 18 
Fraternization 18 
Employment of relatives 19 
Workplace visitors 19 
Solicitation and distribution 20 
Compensation & development 20 
Compensation status 20 
Overtime 21 
Payroll 21 
Performance management 22 
 
 
How we expect managers to lead employees 22 


5.chunking

In [4]:
chunk_size, overlap = 500, 50
if chunk_size <= 0 or not 0 <= overlap < chunk_size: raise ValueError("Require chunk_size > 0 and 0 <= overlap < chunk_size")
chunks=[]; start=0; chunk_id=0
while start < len(text):
    chunk_text=text[start:start+chunk_size].strip()
    if chunk_text: chunks.append({"id":f"chunk-{chunk_id}","text":chunk_text}); chunk_id+=1
    start += chunk_size-overlap
if not chunks: raise ValueError("No PDF text extracted. Scanned PDFs need OCR first.")
print("Total chunks:",len(chunks)); print("First chunk:\n",chunks[0]["text"])


Total chunks: 133
First chunk:
 Employee Handbook 
Welcome 4 
Getting to know our company 4 
Employment basics 5 
Employment contract types 5 
Equal opportunity employment 5 
Recruitment and selection process 6 
Background checks 6 
Referrals 7 
Attendance 8 
Workplace policies 8 
Confidentiality and data protection 8 
Harassment and violence 9 
Workplace harassment 10 
Workplace violence 10 
Workplace safety and health 11 
Preventative action 12 
Emergency management 12 
Smoking 12 
Drug-free workplace 13 
Employee Code o


6. Create local embeddings


In [5]:
import json
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError
OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "embeddinggemma"
GENERATION_MODEL = "gemma3:1b"
def ollama_post(endpoint, payload):
    request=Request(OLLAMA_URL+endpoint,data=json.dumps(payload).encode(),headers={"Content-Type":"application/json"},method="POST")
    try:
        with urlopen(request,timeout=300) as response: return json.loads(response.read().decode())
    except (URLError,HTTPError) as exc:
        raise RuntimeError(f"Ollama call failed. Start Ollama and pull embeddinggemma and {GENERATION_MODEL}. {exc}") from exc
def get_embeddings(texts):
    if not texts: raise ValueError("Cannot embed an empty list.")
    result=ollama_post("/api/embed",{"model":EMBEDDING_MODEL,"input":texts})
    vectors=result.get("embeddings",[])
    if len(vectors)!=len(texts): raise RuntimeError("Ollama returned an unexpected number of embeddings.")
    return vectors
chunk_texts=[chunk["text"] for chunk in chunks]
chunk_embeddings=get_embeddings(chunk_texts)
print("Number of embeddings:",len(chunk_embeddings)); print("Embedding dimensions:",len(chunk_embeddings[0]))


Number of embeddings: 133
Embedding dimensions: 768


7. Prepare in-memory vector store


Chroma is removed; local vector search is performed in memory.


In [6]:
vector_store = list(zip(chunks, chunk_embeddings))
print("Chunks ready for local vector search:", len(vector_store))


Chunks ready for local vector search: 133


Enter your question in the retrieval cell below.


question = input("Ask a question about the PDF: ").strip()
if not question:
    raise ValueError("Question cannot be empty.")
question_embedding = get_embeddings([question])[0]

results = collection.query(
    query_embeddings=[question_embedding],
    n_results=min(3, collection.count()),
)
retrieved_chunks = results["documents"][0]
retrieved_ids = results["ids"][0]
distances = results["distances"][0]
print("Retrieved chunks:\n")
for i, (chunk_id, chunk_text, distance) in enumerate(
    zip(retrieved_ids, retrieved_chunks, distances), start=1
):
    print(f"--- Result {i} | {chunk_id} | distance: {distance:.4f} ---")
    print(chunk_text[:1000])
    print()


In [7]:
import math
question=input("Ask a question about the PDF: ").strip()
if not question: raise ValueError("Question cannot be empty.")
question_embedding=get_embeddings([question])[0]
def cosine_similarity(a,b):
    dot=sum(x*y for x,y in zip(a,b)); na=math.sqrt(sum(x*x for x in a)); nb=math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0
scores=[cosine_similarity(question_embedding,emb) for _,emb in vector_store]
top_indices=sorted(range(len(scores)),key=scores.__getitem__,reverse=True)[:min(3,len(scores))]
retrieved_chunks=[vector_store[i][0]["text"] for i in top_indices]
retrieved_ids=[vector_store[i][0]["id"] for i in top_indices]
for rank,i in enumerate(top_indices,start=1):
    print(f"--- Result {rank} | {retrieved_ids[rank-1]} | similarity: {scores[i]:.4f} ---")
    print(retrieved_chunks[rank-1][:1000],"\n")


--- Result 1 | chunk-19 | similarity: 0.6381 ---
takeholders alike. These policies help us build a 
productive, lawful and pleasant workplace.  
Confidentiality and data protection 
We want to ensure that private information about clients, employees, partners and our 
company is well-protected. Examples of confidential information are: 
 
 
 
■ Employee records 
■ Unpublished financial information 
■ Data of customers/partners/vendors 
■ Customer lists (existing and prospective) 
■ Unpublished goals, forecasts and initiatives marked as confide 

--- Result 2 | chunk-18 | similarity: 0.6258 ---
r scheduled working hours. If you face an 
emergency that prevents you from coming to work one day, contact your manager as 
soon as possible. We will excuse unreported absences in cases of [serious accidents, 
acute medical emergencies.] But, whenever possible, we should know when you won’t 
be coming in. 
Workplace policies 
This section describes policies that apply to everyone at our company:

context = "\n\n--- Retrieved chunk ---\n\n".join(retrieved_chunks)
prompt = f"""Answer the user's question using only the retrieved context below. If the answer is not present in the context, say that the information is not available in the provided document.

Retrieved context:
{context}

User question: {question}"""
print(prompt)


In [8]:
context="\n\n--- Retrieved chunk ---\n\n".join(retrieved_chunks)
prompt=f"""Answer the user's question using only the retrieved context below. If the answer is not present in the context, say that the information is not available in the provided document.

Retrieved context:
{context}

User question: {question}"""
print(prompt)


Answer the user's question using only the retrieved context below. If the answer is not present in the context, say that the information is not available in the provided document.

Retrieved context:
takeholders alike. These policies help us build a 
productive, lawful and pleasant workplace.  
Confidentiality and data protection 
We want to ensure that private information about clients, employees, partners and our 
company is well-protected. Examples of confidential information are: 
 
 
 
■ Employee records 
■ Unpublished financial information 
■ Data of customers/partners/vendors 
■ Customer lists (existing and prospective) 
■ Unpublished goals, forecasts and initiatives marked as confide

--- Retrieved chunk ---

r scheduled working hours. If you face an 
emergency that prevents you from coming to work one day, contact your manager as 
soon as possible. We will excuse unreported absences in cases of [serious accidents, 
acute medical emergencies.] But, whenever possible, we should 

8. Generate answer with local Ollama model


In [9]:
response=ollama_post("/api/generate",{"model":GENERATION_MODEL,"prompt":prompt,"stream":False})
answer=response.get("response","")
print("Answer:\n")
print(answer)


Answer:

This section describes policies that apply to everyone at our company: employees, contractors, volunteers, vendors and stakeholders alike. These policies help us build a productive, lawful and pleasant workplace.
